<a href="https://colab.research.google.com/github/ondrekm/stanford-cars-model-comparison/blob/main/%C3%96sszehasonl%C3%ADt%C3%A1s.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# KÓD #0 – KÖRNYEZET ÉS ALAPBEÁLLÍTÁSOK

import os, re, math, random, time
from dataclasses import dataclass
from typing import Tuple, Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import StanfordCars

import timm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

!pip -q install grad-cam

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

@dataclass
class CFG:
    data_root: str = "/content"
    drive_root: str = "/content/drive/MyDrive/stanford_cars_project"
    save_root: str = "/content/drive/MyDrive/stanford_cars_project/data"
    model_root: str = "/content/drive/MyDrive/stanford_cars_project/models"
    output_root: str = "/content/drive/MyDrive/stanford_cars_project/outputs"
    batch_size: int = 8
    num_workers: int = 0
    lr: float = 3e-4
    epochs: int = 10
    img_size: int = 224
    model_name: str = "resnet50"
    val_ratio: float = 0.2

cfg = CFG()

os.makedirs(cfg.save_root, exist_ok=True)
os.makedirs(cfg.model_root, exist_ok=True)
os.makedirs(cfg.output_root, exist_ok=True)

print("SAVE:", cfg.save_root)
print("MODELS:", cfg.model_root)
print("OUTPUT:", cfg.output_root)

In [ ]:
# Opcionális – dataset letöltése Kaggle-ről (csak első alkalommal szükséges)

import os

DATA_ROOT = cfg.data_root
DATASET_DIR = os.path.join(DATA_ROOT, "stanford_cars")

if not os.path.exists("/content/kaggle.json"):
    raise FileNotFoundError(
        "Hiányzik a /content/kaggle.json. "
        "Töltsd fel a Kaggle API kulcsot a Colab Files panelen."
    )

!mkdir -p /root/.kaggle
!cp /content/kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

if not os.path.isdir(DATASET_DIR):
    print("Stanford Cars dataset nem található, letöltés indul...")
    try:
        import kaggle
    except ImportError:
        !pip -q install kaggle
        import kaggle

    !mkdir -p "{DATA_ROOT}"
    !kaggle datasets download -d emanuelriquelmem/stanford-cars-pytorch -p "{DATA_ROOT}" --unzip
    print("Letöltés kész.")
else:
    print("Stanford Cars dataset már megvan, letöltés kihagyva.")

print("DATA_ROOT:", DATA_ROOT)
print("Tartalom:", os.listdir(DATA_ROOT))

# Új szakasz

In [ ]:
# KÓD #1 – ADATBETÖLTÉS (STANFORD CARS) + TRANSFORMS

from torchvision import transforms

def build_transforms(img_size=224, color_jitter=0.1, use_affine=False, use_erasing=False):
    train_list = [
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(
            brightness=color_jitter,
            contrast=color_jitter,
            saturation=color_jitter
        ),
    ]

    if use_affine:
        train_list.append(
            transforms.RandomAffine(
                degrees=8,
                translate=(0.03, 0.03),
                scale=(0.97, 1.03)
            )
        )

    train_list.extend([
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406),
                             std=(0.229, 0.224, 0.225)),
    ])

    if use_erasing:
        train_list.append(transforms.RandomErasing(p=0.25, scale=(0.02, 0.10)))

    train_tfms = transforms.Compose(train_list)

    val_tfms = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406),
                             std=(0.229, 0.224, 0.225)),
    ])

    return train_tfms, val_tfms









def assert_stanfordcars_present(root: str):
    base = os.path.join(root, "stanford_cars")
    required_dirs = [
        os.path.join(base, "cars_train"),
        os.path.join(base, "cars_test"),
        os.path.join(base, "devkit"),
    ]
    missing = [d for d in required_dirs if not os.path.isdir(d)]

    if missing:
        print("HIBA: A Stanford Cars dataset nincs a várt helyen.")
        print("Várt hely:", base)
        print("Hiányzó mappák:")
        for d in missing:
            print(" -", d)
        print("\nTeendő:")
        print("1) Ellenőrizd, hogy a dataset át lett-e másolva a /content alá.")
        print("2) Ellenőrizd, hogy a mappaszerkezet megfelelő-e.")
        raise FileNotFoundError("StanfordCars dataset not found in expected folder structure.")


def load_stanford_cars(root: str, train_tfms, val_tfms):
    assert_stanfordcars_present(root)

    train_ds = StanfordCars(
        root=root,
        split="train",
        download=False,
        transform=train_tfms
    )

    test_ds = StanfordCars(
        root=root,
        split="test",
        download=False,
        transform=val_tfms
    )

    return train_ds, test_ds


# próba betöltés
train_tfms, val_tfms = build_transforms(img_size=cfg.img_size)
train_ds_full, test_ds = load_stanford_cars(cfg.data_root, train_tfms, val_tfms)

print("Train size:", len(train_ds_full), "Test size:", len(test_ds), "Classes:", len(train_ds_full.classes))
print("Példa class:", train_ds_full.classes[0])

In [ ]:
# KÓD #2 – FUTÁSI KONFIGURÁCIÓ + SEED + DATALOADEREK

from dataclasses import dataclass
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from sklearn.model_selection import train_test_split
import numpy as np
import random
import torch

@dataclass
class RunConfig:
    run_name: str
    seed: int
    model_name: str = "resnet50"
    epochs: int = 5
    batch_size: int = 8
    num_workers: int = 0
    lr: float = 3e-4
    img_size: int = 224
    val_ratio: float = 0.2
    train_fraction: float = 1.0
    color_jitter: float = 0.1
    use_affine: bool = False
    use_erasing: bool = False
    weighted_sampling: bool = False

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def build_loaders(data_root: str, run_cfg: RunConfig):
    set_seed(run_cfg.seed)

    train_tfms, val_tfms = build_transforms(
        img_size=run_cfg.img_size,
        color_jitter=run_cfg.color_jitter,
        use_affine=run_cfg.use_affine,
        use_erasing=run_cfg.use_erasing
    )

    # Train dataset train transzformmal
    train_ds_full = StanfordCars(
        root=data_root,
        split="train",
        download=False,
        transform=train_tfms
    )

    # Train split, validációs transzformmal
    val_base_ds = StanfordCars(
        root=data_root,
        split="train",
        download=False,
        transform=val_tfms
    )

    # Test dataset validációs / teszt transzformmal
    test_ds = StanfordCars(
        root=data_root,
        split="test",
        download=False,
        transform=val_tfms
    )

    if hasattr(train_ds_full, "labels"):
        labels = np.array(train_ds_full.labels)
    else:
        labels = np.array([train_ds_full[i][1] for i in range(len(train_ds_full))])

    indices = np.arange(len(train_ds_full))

    train_idx, val_idx = train_test_split(
        indices,
        test_size=run_cfg.val_ratio,
        random_state=run_cfg.seed,
        stratify=labels
    )

    # Ha csak a train egy részét akarom használni
    if run_cfg.train_fraction < 1.0:
        train_labels = labels[train_idx]
        train_idx, _ = train_test_split(
            train_idx,
            train_size=run_cfg.train_fraction,
            random_state=run_cfg.seed,
            stratify=train_labels
        )

    train_ds = Subset(train_ds_full, train_idx)
    val_ds = Subset(val_base_ds, val_idx)

    sampler = None
    shuffle = True

    if run_cfg.weighted_sampling:
        train_targets = labels[train_idx]
        class_counts = np.bincount(train_targets)
        class_weights = 1.0 / np.maximum(class_counts, 1)
        sample_weights = class_weights[train_targets]

        sampler = WeightedRandomSampler(
            weights=torch.DoubleTensor(sample_weights),
            num_samples=len(sample_weights),
            replacement=True
        )
        shuffle = False

    train_loader = DataLoader(
        train_ds,
        batch_size=run_cfg.batch_size,
        shuffle=shuffle if sampler is None else False,
        sampler=sampler,
        num_workers=run_cfg.num_workers,
        pin_memory=torch.cuda.is_available()
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=run_cfg.batch_size,
        shuffle=False,
        num_workers=run_cfg.num_workers,
        pin_memory=torch.cuda.is_available()
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=run_cfg.batch_size,
        shuffle=False,
        num_workers=run_cfg.num_workers,
        pin_memory=torch.cuda.is_available()
    )

    print(
        f"[{run_cfg.run_name}] "
        f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)} | Test samples: {len(test_ds)}"
    )
    print(
        f"[{run_cfg.run_name}] "
        f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}"
    )

    return train_loader, val_loader, test_loader, train_ds_full

In [ ]:
# KÓD #3 – MODELLÉPÍTÉS + TRAIN / VAL / TEST + TELJES FUTTATÁS

import time
import pandas as pd
import torch.nn as nn
import torch.optim as optim
import timm

def build_model(model_name: str, num_classes: int, device):
    model = timm.create_model(
        model_name,
        pretrained=True,
        num_classes=num_classes
    ).to(device)
    return model

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    losses = []
    correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        preds = torch.argmax(logits, dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return float(np.mean(losses)), float(correct / total)

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    losses = []
    correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = criterion(logits, y)

        losses.append(loss.item())
        preds = torch.argmax(logits, dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return float(np.mean(losses)), float(correct / total)

@torch.no_grad()
def evaluate_test(model, loader, device):
    model.eval()
    correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return float(correct / total)

def run_experiment(run_cfg, data_root=None, device=None):
    if data_root is None:
        data_root = cfg.data_root
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    set_seed(run_cfg.seed)

    train_loader, val_loader, test_loader, train_ds_full = build_loaders(data_root, run_cfg)

    num_classes = len(train_ds_full.classes)
    model = build_model(run_cfg.model_name, num_classes, device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=run_cfg.lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=run_cfg.epochs
    )

    history = []
    best_val_acc = -1.0
    best_ckpt_path = os.path.join(
        cfg.model_root,
        f"best_{run_cfg.run_name}_{run_cfg.model_name}.pt"
    )
    start = time.time()

    for epoch in range(1, run_cfg.epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        va_loss, va_acc = evaluate(model, val_loader, criterion, device)

        scheduler.step()
        lr = optimizer.param_groups[0]["lr"]

        history.append({
            "run_name": run_cfg.run_name,
            "seed": run_cfg.seed,
            "model_name": run_cfg.model_name,
            "epoch": epoch,
            "train_loss": tr_loss,
            "train_acc": tr_acc,
            "val_loss": va_loss,
            "val_acc": va_acc,
            "lr": lr
        })

        print(
            f"[{run_cfg.run_name}] "
            f"[{epoch:02d}/{run_cfg.epochs}] "
            f"train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | "
            f"val_loss={va_loss:.4f} val_acc={va_acc:.4f} | "
            f"lr={lr:.2e}"
        )

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            torch.save(model.state_dict(), best_ckpt_path)

    elapsed_min = (time.time() - start) / 60.0

    model.load_state_dict(torch.load(best_ckpt_path, map_location=device))
    test_acc = evaluate_test(model, test_loader, device)

    hist_df = pd.DataFrame(history)
    hist_csv_name = os.path.join(
        cfg.save_root,
        f"history_{run_cfg.run_name}_{run_cfg.model_name}.csv"
    )
    hist_df.to_csv(hist_csv_name, index=False)

    result = {
        "run_name": run_cfg.run_name,
        "seed": run_cfg.seed,
        "model_name": run_cfg.model_name,
        "epochs": run_cfg.epochs,
        "batch_size": run_cfg.batch_size,
        "lr": run_cfg.lr,
        "img_size": run_cfg.img_size,
        "train_fraction": run_cfg.train_fraction,
        "color_jitter": run_cfg.color_jitter,
        "use_affine": run_cfg.use_affine,
        "use_erasing": run_cfg.use_erasing,
        "weighted_sampling": run_cfg.weighted_sampling,
        "best_val_acc": best_val_acc,
        "test_acc": test_acc,
        "time_min": elapsed_min,
        "checkpoint": best_ckpt_path,
        "history_csv": hist_csv_name
    }

    print(
        f"[{run_cfg.run_name}] Done. "
        f"Best val acc: {best_val_acc:.4f} | "
        f"Test acc: {test_acc:.4f} | "
        f"Time: {elapsed_min:.1f} min"
    )

    return result, hist_df

In [ ]:
# KÓD #4 – BLOKKOS FUTTATÓ SEGÉDFÜGGVÉNY

import os
import pandas as pd

def run_block(runs, block_name, save_dir=None):
    if save_dir is None:
        save_dir = cfg.save_root

    all_results = []
    all_histories = []

    for run_cfg in runs:
        print("\n" + "=" * 80)
        print(f"FUTTATÁS: {run_cfg.run_name} | model={run_cfg.model_name} | seed={run_cfg.seed}")
        print("=" * 80)

        result, hist_df = run_experiment(run_cfg, data_root=cfg.data_root, device=device)
        all_results.append(result)
        all_histories.append(hist_df)

    results_df = pd.DataFrame(all_results)
    histories_df = pd.concat(all_histories, ignore_index=True)

    results_path = os.path.join(save_dir, f"{block_name}_results.csv")
    histories_path = os.path.join(save_dir, f"{block_name}_histories.csv")

    results_df.to_csv(results_path, index=False)
    histories_df.to_csv(histories_path, index=False)

    print(f"\nMentve: {results_path}")
    print(f"Mentve: {histories_path}")

    return results_df, histories_df

In [ ]:
# KÓD #4 – BLOKK 1

RUNS_BLOCK1 = [
    RunConfig(
        run_name="baseline_s42",
        seed=42,
        model_name="resnet50",
        epochs=5,
        batch_size=8,
        num_workers=0
    ),
    RunConfig(
        run_name="baseline_s43",
        seed=43,
        model_name="resnet50",
        epochs=5,
        batch_size=8,
        num_workers=0
    ),
    RunConfig(
        run_name="baseline_s44",
        seed=44,
        model_name="resnet50",
        epochs=5,
        batch_size=8,
        num_workers=0
    ),
]

results_block1, histories_block1 = run_block(
    RUNS_BLOCK1,
    "block1",
    save_dir=cfg.save_root
)

display(results_block1)

In [ ]:

# KÓD #4 – BLOKK 2
RUNS_BLOCK2 = [
    RunConfig(
        run_name="baseline_s45",
        seed=45,
        model_name="resnet50",
        epochs=2,
        batch_size=2,
        num_workers=0,
        img_size=160,
        train_fraction=0.2
    ),
    RunConfig(
        run_name="baseline_s46",
        seed=46,
        model_name="resnet50",
        epochs=2,
        batch_size=2,
        num_workers=0,
        img_size=160,
        train_fraction=0.2
    ),
    RunConfig(
        run_name="jit02",
        seed=42,
        model_name="resnet50",
        epochs=2,
        batch_size=2,
        num_workers=0,
        img_size=160,
        train_fraction=0.2,
        color_jitter=0.2
    ),
]

results_block2, histories_block2 = run_block(
    RUNS_BLOCK2,
    "block2",
    save_dir=cfg.save_root
)

display(results_block2)

In [ ]:
# KÓD #4 – BLOKK 3
RUNS_BLOCK3 = [
    RunConfig(
        run_name="affine",
        seed=42,
        model_name="resnet50",
        epochs=2,
        batch_size=2,
        num_workers=0,
        img_size=160,
        train_fraction=0.2,
        use_affine=True
    ),
    RunConfig(
        run_name="erasing",
        seed=42,
        model_name="resnet50",
        epochs=2,
        batch_size=2,
        num_workers=0,
        img_size=160,
        train_fraction=0.2,
        use_erasing=True
    ),
    RunConfig(
        run_name="weighted",
        seed=42,
        model_name="resnet50",
        epochs=2,
        batch_size=2,
        num_workers=0,
        img_size=160,
        train_fraction=0.2,
        weighted_sampling=True
    ),
    RunConfig(
        run_name="full_aug",
        seed=42,
        model_name="resnet50",
        epochs=2,
        batch_size=2,
        num_workers=0,
        img_size=160,
        train_fraction=0.2,
        color_jitter=0.2,
        use_affine=True,
        use_erasing=True
    ),
]

results_block3, histories_block3 = run_block(
    RUNS_BLOCK3,
    "block3",
    save_dir=cfg.save_root
)

display(results_block3)

In [ ]:
# KÓD #4 – BLOKK 4 (REPEATABILITY / STABILITY)

RUNS_BLOCK4 = [
    RunConfig(
        run_name="baseline_rep_s42",
        seed=42,
        model_name="resnet50",
        epochs=5,
        batch_size=8,
        num_workers=0
    ),
    RunConfig(
        run_name="baseline_rep_s43",
        seed=43,
        model_name="resnet50",
        epochs=5,
        batch_size=8,
        num_workers=0
    ),
    RunConfig(
        run_name="baseline_rep_s44",
        seed=44,
        model_name="resnet50",
        epochs=5,
        batch_size=8,
        num_workers=0
    ),
    RunConfig(
        run_name="full_aug_rep_s42",
        seed=42,
        model_name="resnet50",
        epochs=2,
        batch_size=2,
        num_workers=0,
        img_size=160,
        train_fraction=0.2,
        color_jitter=0.2,
        use_affine=True,
        use_erasing=True
    ),
    RunConfig(
        run_name="full_aug_rep_s43",
        seed=43,
        model_name="resnet50",
        epochs=2,
        batch_size=2,
        num_workers=0,
        img_size=160,
        train_fraction=0.2,
        color_jitter=0.2,
        use_affine=True,
        use_erasing=True
    ),
    RunConfig(
        run_name="full_aug_rep_s44",
        seed=44,
        model_name="resnet50",
        epochs=2,
        batch_size=2,
        num_workers=0,
        img_size=160,
        train_fraction=0.2,
        color_jitter=0.2,
        use_affine=True,
        use_erasing=True
    ),
]
results_block4, histories_block4 = run_block(
    RUNS_BLOCK4,
    "block4",
    save_dir=cfg.save_root
)

display(results_block4)


In [ ]:
# KÓD #4 – BLOKKOK EGYESÍTÉSE

import os
import pandas as pd

result_files = [
    os.path.join(cfg.save_root, "block1_results.csv"),
    os.path.join(cfg.save_root, "block2_results.csv"),
    os.path.join(cfg.save_root, "block3_results.csv"),
    os.path.join(cfg.save_root, "block4_results.csv"),
]

history_files = [
    os.path.join(cfg.save_root, "block1_histories.csv"),
    os.path.join(cfg.save_root, "block2_histories.csv"),
    os.path.join(cfg.save_root, "block3_histories.csv"),
    os.path.join(cfg.save_root, "block4_histories.csv"),
]

missing_results = [f for f in result_files if not os.path.exists(f)]
missing_histories = [f for f in history_files if not os.path.exists(f)]

if missing_results:
    raise FileNotFoundError(f"Hiányzó result fájl(ok): {missing_results}")

if missing_histories:
    raise FileNotFoundError(f"Hiányzó history fájl(ok): {missing_histories}")

results_df = pd.concat(
    [pd.read_csv(f) for f in result_files],
    ignore_index=True
)

all_histories_df = pd.concat(
    [pd.read_csv(f) for f in history_files],
    ignore_index=True
)

results_df.to_csv(
    os.path.join(cfg.save_root, "multi_run_results_10.csv"),
    index=False
)

all_histories_df.to_csv(
    os.path.join(cfg.save_root, "multi_run_histories_10.csv"),
    index=False
)

print("Mentve:")
print(os.path.join(cfg.save_root, "multi_run_results_10.csv"))
print(os.path.join(cfg.save_root, "multi_run_histories_10.csv"))

display(results_df)

In [ ]:
# MULTI-RUN EREDMÉNYEK ÖSSZEGZÉSE

summary_df = (
    results_df.groupby("model_name")[["best_val_acc", "test_acc", "time_min"]]
    .agg(["mean", "std", "min", "max"])
)

display(summary_df)

In [ ]:
# LEGJOBB FUTÁSOK

best_runs_df = results_df.sort_values(by="test_acc", ascending=False).reset_index(drop=True)
display(best_runs_df.head(10))

In [ ]:
# VARIÁCIÓK HATÁSA – rövid összehasonlítás

cols_to_show = [
    "run_name", "seed", "model_name",
    "epochs", "batch_size", "lr", "img_size",
    "train_fraction", "color_jitter",
    "use_affine", "use_erasing", "weighted_sampling",
    "best_val_acc", "test_acc", "time_min"
]

variation_df = results_df[cols_to_show].sort_values(by="test_acc", ascending=False).reset_index(drop=True)
display(variation_df)

In [ ]:
# KÓD #5 – EMBEDDING KINYERÉS A LEGJOBB FUTÁSBÓL

import os
import numpy as np
import torch
import timm
from torchvision.datasets import StanfordCars
from torch.utils.data import DataLoader

def load_model_for_embeddings(model_name: str, ckpt_path: str, num_classes: int, device):
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Nem találom a checkpointot: {ckpt_path}")

    model = timm.create_model(
        model_name,
        pretrained=False,
        num_classes=num_classes
    ).to(device)

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()
    return model

@torch.no_grad()
def extract_embeddings(model, loader, device, max_items=None):
    feats_list = []
    labels_list = []
    seen = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)

        feats = model.forward_features(x)

        if feats.ndim == 4:
            emb = feats.mean(dim=(2, 3))
        else:
            emb = feats

        feats_list.append(emb.cpu().numpy())
        labels_list.append(y.numpy())

        seen += x.size(0)
        if max_items is not None and seen >= max_items:
            break

    X = np.concatenate(feats_list, axis=0)
    y = np.concatenate(labels_list, axis=0)

    if max_items is not None:
        X, y = X[:max_items], y[:max_items]

    return X, y

# Legjobb futás kiválasztása
best_row = results_df.sort_values("test_acc", ascending=False).iloc[0]
best_model_name = best_row["model_name"]
best_ckpt_path = best_row["checkpoint"]

print("Embedding elemzéshez kiválasztott futás:")
print(best_row[["run_name", "model_name", "test_acc", "checkpoint"]])

# Teszt transzform
_, val_tfms = build_transforms(img_size=int(best_row["img_size"]))

# Teszt dataset
test_ds_embed = StanfordCars(
    root=cfg.data_root,
    split="test",
    download=False,
    transform=val_tfms
)

test_loader_embed = DataLoader(
    test_ds_embed,
    batch_size=32 if device.type == "cpu" else 64,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

num_classes = len(test_ds_embed.classes)

model_embed = load_model_for_embeddings(
    model_name=best_model_name,
    ckpt_path=best_ckpt_path,
    num_classes=num_classes,
    device=device
)

max_items_embed = 1000 if device.type == "cpu" else 3000

X, y_true = extract_embeddings(
    model=model_embed,
    loader=test_loader_embed,
    device=device,
    max_items=max_items_embed
)

print("Embeddings shape:", X.shape, "| labels shape:", y_true.shape)

In [ ]:
# KÓD #6 – PCA + KMeans + vizualizáció + sziluett mutató

import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

def run_clustering_analysis(X, n_clusters=8, random_state=42, model_name="model"):
    scaler = StandardScaler()
    Xz = scaler.fit_transform(X)

    pca = PCA(n_components=2, random_state=random_state)
    X2 = pca.fit_transform(Xz)

    print("PCA explained variance ratio:", pca.explained_variance_ratio_)
    print("PCA összes magyarázott variancia:", pca.explained_variance_ratio_.sum())

    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    clusters = kmeans.fit_predict(X2)

    sil_score = silhouette_score(X2, clusters)
    print(f"Sziluett mutató (K={n_clusters}): {sil_score:.4f}")

    plt.figure(figsize=(10, 7))
    plt.scatter(X2[:, 0], X2[:, 1], s=8, c=clusters)
    plt.title(f"KMeans klaszterek (K={n_clusters}) – {model_name} embeddings + PCA(2D)")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.grid(True)
    plt.show()

    return {
        "X2": X2,
        "clusters": clusters,
        "pca": pca,
        "kmeans": kmeans,
        "silhouette": sil_score
    }

def evaluate_k_values(X2, Ks=(4, 6, 8, 10, 12)):
    scores = []

    for k in Ks:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X2)
        score = silhouette_score(X2, labels)
        scores.append(score)
        print(f"K={k} -> silhouette={score:.4f}")

    plt.figure(figsize=(8, 5))
    plt.plot(Ks, scores, marker="o")
    plt.xlabel("K (klaszterszám)")
    plt.ylabel("Sziluett mutató")
    plt.title("Klaszterszám hatása a klaszterezés minőségére")
    plt.grid(True)
    plt.show()

    return Ks, scores

clustering_result = run_clustering_analysis(
    X,
    n_clusters=8,
    model_name=best_model_name
)

X2 = clustering_result["X2"]
Ks, scores = evaluate_k_values(X2)

In [ ]:

clustering_result = run_clustering_analysis(
    X,
    n_clusters=8,
    model_name=best_model_name
)

X2 = clustering_result["X2"]

Ks, scores = evaluate_k_values(X2)

In [ ]:
# KÓD #7 – COSINE SIMILARITY + TOP-K RETRIEVAL

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from torch.utils.data import Subset

def normalize_embeddings(X):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-12)
    return X / norms

def compute_similarity_matrix(X):
    Xn = normalize_embeddings(X)
    return cosine_similarity(Xn)

def topk_retrieval(sim_matrix, labels, ks=(1, 5)):
    n = sim_matrix.shape[0]
    sim = sim_matrix.copy()
    np.fill_diagonal(sim, -np.inf)

    results = {}
    for k in ks:
        correct = 0
        for i in range(n):
            topk_idx = np.argsort(sim[i])[::-1][:k]
            topk_labels = labels[topk_idx]
            if labels[i] in topk_labels:
                correct += 1
        results[f"top_{k}"] = correct / n

    return results

def show_retrieval_examples(sim_matrix, labels, dataset, class_names=None, n_queries=5, top_k=5, seed=42):
    rng = np.random.default_rng(seed)
    n = len(labels)
    query_indices = rng.choice(n, size=min(n_queries, n), replace=False)

    sim = sim_matrix.copy()
    np.fill_diagonal(sim, -np.inf)

    fig, axes = plt.subplots(len(query_indices), top_k + 1, figsize=(3 * (top_k + 1), 3 * len(query_indices)))

    if len(query_indices) == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, q_idx in enumerate(query_indices):
        top_idx = np.argsort(sim[q_idx])[::-1][:top_k]

        q_img, q_label = dataset[q_idx]
        q_img_np = q_img.permute(1, 2, 0).cpu().numpy()
        q_img_np = np.clip(q_img_np, 0, 1)

        axes[row, 0].imshow(q_img_np)
        q_title = f"Query\nlabel={q_label}"
        if class_names is not None:
            q_title = f"Query\n{class_names[q_label]}"
        axes[row, 0].set_title(q_title, fontsize=9)
        axes[row, 0].axis("off")

        for col, idx in enumerate(top_idx, start=1):
            img, lab = dataset[idx]
            img_np = img.permute(1, 2, 0).cpu().numpy()
            img_np = np.clip(img_np, 0, 1)
            sim_val = sim[q_idx, idx]

            title = f"label={lab}\nsim={sim_val:.3f}"
            if class_names is not None:
                title = f"{class_names[lab]}\nsim={sim_val:.3f}"

            axes[row, col].imshow(img_np)
            axes[row, col].set_title(title, fontsize=8)
            axes[row, col].axis("off")

    plt.tight_layout()
    plt.show()

sim_matrix = compute_similarity_matrix(X)

retrieval_results = topk_retrieval(sim_matrix, y_true, ks=(1, 5))
print("Top-k eredmények:")
for k, v in retrieval_results.items():
    print(f"{k}: {v:.4f}")

viz_tfms = transforms.Compose([
    transforms.Resize((int(best_row["img_size"]), int(best_row["img_size"]))),
    transforms.ToTensor(),
])

test_ds_viz = StanfordCars(
    root=cfg.data_root,
    split="test",
    download=False,
    transform=viz_tfms
)

max_items_viz = len(y_true)

show_retrieval_examples(
    sim_matrix=sim_matrix[:max_items_viz, :max_items_viz],
    labels=y_true,
    dataset=Subset(test_ds_viz, range(max_items_viz)),
    class_names=test_ds_viz.classes,
    n_queries=5,
    top_k=5,
    seed=42
)

In [ ]:
# KÓD 7/B– GRAD-CAM  SEGÉDFÜGGVÉNYEK

def build_gradcam_model(model_name: str, ckpt_path: str, num_classes: int, device):
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Nem találom a checkpointot: {ckpt_path}")

    model = timm.create_model(
        model_name,
        pretrained=False,
        num_classes=num_classes
    ).to(device)

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    if "resnet" in model_name:
        target_layers = [model.layer4[-1]]
    else:
        raise ValueError(f"Ehhez a modellhez nincs még beállítva target layer: {model_name}")

    cam = GradCAM(model=model, target_layers=target_layers)
    return model, cam


def denorm_image(img_t, mean, std):
    img = img_t.detach().cpu().numpy().transpose(1, 2, 0)
    img = (img * std) + mean
    img = np.clip(img, 0.0, 1.0)
    return img




In [ ]:
# KÓD #7/C – GRAD-CAM VIZUALIZÁCIÓS FÜGGVÉNY

def generate_gradcam_examples(
    model,
    cam,
    dataset,
    device,
    class_names=None,
    n_examples=5,
    seed=42,
    save_dir=None
):
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)

    rng = np.random.default_rng(seed)
    idxs = rng.choice(len(dataset), size=min(n_examples, len(dataset)), replace=False)

    fig, axes = plt.subplots(len(idxs), 3, figsize=(12, 4 * len(idxs)))
    if len(idxs) == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, idx in enumerate(idxs):
        img_t, label = dataset[idx]
        input_tensor = img_t.unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(input_tensor)
            pred = torch.argmax(logits, dim=1).item()

        targets = [ClassifierOutputTarget(pred)]
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]

        rgb_img = denorm_image(img_t, mean, std)
        cam_img = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

        axes[row, 0].imshow(rgb_img)
        axes[row, 0].set_title("Eredeti kép")
        axes[row, 0].axis("off")

        axes[row, 1].imshow(grayscale_cam, cmap="jet")
        axes[row, 1].set_title("Grad-CAM hőtérkép")
        axes[row, 1].axis("off")

        title = f"Valós: {label} | Pred: {pred}"
        if class_names is not None:
            title = f"Valós: {class_names[label]}\nPred: {class_names[pred]}"

        axes[row, 2].imshow(cam_img)
        axes[row, 2].set_title(title)
        axes[row, 2].axis("off")

        if save_dir is not None:
            fname_base = f"sample_{row}_idx_{idx}"
            plt.imsave(os.path.join(save_dir, f"{fname_base}_original.png"), rgb_img)
            plt.imsave(os.path.join(save_dir, f"{fname_base}_heatmap.png"), grayscale_cam, cmap="jet")
            plt.imsave(os.path.join(save_dir, f"{fname_base}_overlay.png"), cam_img)

    plt.tight_layout()

    if save_dir is not None:
        plt.savefig(os.path.join(save_dir, "gradcam_grid.png"))

    plt.show()

    if save_dir is not None:
        print(f"Grad-CAM képek mentve ide: {save_dir}")

In [ ]:
#csak első futáskor
print(best_model_name)
print(best_ckpt_path)
print(os.path.exists(best_ckpt_path))

In [ ]:
# KÓD #7/D – GRAD-CAM FUTTATÁS

# Ugyanazzal a képmérettel dolgozok, mint a kiválasztott modell
_, val_tfms_cam = build_transforms(img_size=int(best_row["img_size"]))

test_ds_cam = StanfordCars(
    root=cfg.data_root,
    split="test",
    download=False,
    transform=val_tfms_cam
)

num_classes_cam = len(test_ds_cam.classes)

gradcam_model, gradcam = build_gradcam_model(
    model_name=best_model_name,
    ckpt_path=best_ckpt_path,
    num_classes=num_classes_cam,
    device=device
)

gradcam_save_dir = os.path.join(cfg.output_root, "gradcam_outputs")

generate_gradcam_examples(
    model=gradcam_model,
    cam=gradcam,
    dataset=test_ds_cam,
    device=device,
    class_names=test_ds_cam.classes,
    n_examples=5,
    seed=42,
    save_dir=gradcam_save_dir
)

In [ ]:
# KÓD #8 – EREDMÉNYEK ÖSSZESÍTÉSE ÉS MENTÉSE

import pandas as pd
import numpy as np
import os

def save_experiment_outputs(
    results_df,
    clustering_result,
    retrieval_results,
    output_prefix="final_results"
):
    output_dir = cfg.output_root

    results_csv = os.path.join(output_dir, f"{output_prefix}_multi_run.csv")
    results_df.to_csv(results_csv, index=False)

    summary_df = (
        results_df.groupby("model_name")[["best_val_acc", "test_acc", "time_min"]]
        .agg(["mean", "std", "min", "max"])
    )
    summary_csv = os.path.join(output_dir, f"{output_prefix}_summary.csv")
    summary_df.to_csv(summary_csv)

    clustering_df = pd.DataFrame({
        "pc1": clustering_result["X2"][:, 0],
        "pc2": clustering_result["X2"][:, 1],
        "cluster": clustering_result["clusters"]
    })
    clustering_csv = os.path.join(output_dir, f"{output_prefix}_clustering.csv")
    clustering_df.to_csv(clustering_csv, index=False)

    retrieval_df = pd.DataFrame([retrieval_results])
    retrieval_csv = os.path.join(output_dir, f"{output_prefix}_retrieval.csv")
    retrieval_df.to_csv(retrieval_csv, index=False)

    print("Mentett fájlok:")
    print(" -", results_csv)
    print(" -", summary_csv)
    print(" -", clustering_csv)
    print(" -", retrieval_csv)

    return {
        "results_csv": results_csv,
        "summary_csv": summary_csv,
        "clustering_csv": clustering_csv,
        "retrieval_csv": retrieval_csv,
        "summary_df": summary_df,
        "retrieval_df": retrieval_df,
        "clustering_df": clustering_df
    }

saved_outputs = save_experiment_outputs(
    results_df=results_df,
    clustering_result=clustering_result,
    retrieval_results=retrieval_results,
    output_prefix="stanford_cars_experiment"
)

print("\n=== MULTI-RUN EREDMÉNYEK ===")
display(results_df.sort_values(by="test_acc", ascending=False))

print("\n=== MODELL SZINTŰ ÖSSZEGZÉS ===")
display(saved_outputs["summary_df"])

print("\n=== RETRIEVAL EREDMÉNYEK ===")
display(saved_outputs["retrieval_df"])

best_row = results_df.sort_values(by="test_acc", ascending=False).iloc[0]
print("Legjobb futás:")
print(f"run_name : {best_row['run_name']}")
print(f"model_name : {best_row['model_name']}")
print(f"best_val_acc : {best_row['best_val_acc']:.4f}")
print(f"test_acc : {best_row['test_acc']:.4f}")
print(f"time_min : {best_row['time_min']:.2f}")

print("\nRetrieval eredmények:")
for k, v in retrieval_results.items():
    print(f"{k}: {v:.4f}")

print(f"\nSziluett mutató: {clustering_result['silhouette']:.4f}")

In [ ]:
saved_outputs = save_experiment_outputs(
    results_df=results_df,
    clustering_result=clustering_result,
    retrieval_results=retrieval_results,
    output_prefix="stanford_cars_experiment"
)

In [ ]:
print("\n=== MULTI-RUN EREDMÉNYEK ===")
display(results_df.sort_values(by="test_acc", ascending=False))

print("\n=== MODELL SZINTŰ ÖSSZEGZÉS ===")
display(saved_outputs["summary_df"])

print("\n=== RETRIEVAL EREDMÉNYEK ===")
display(saved_outputs["retrieval_df"])

In [ ]:
best_row = results_df.sort_values(by="test_acc", ascending=False).iloc[0]

print("Legjobb futás:")
print(f"run_name     : {best_row['run_name']}")
print(f"model_name   : {best_row['model_name']}")
print(f"best_val_acc : {best_row['best_val_acc']:.4f}")
print(f"test_acc     : {best_row['test_acc']:.4f}")
print(f"time_min     : {best_row['time_min']:.2f}")

print("\nRetrieval eredmények:")
for k, v in retrieval_results.items():
    print(f"{k}: {v:.4f}")

print(f"\nSziluett mutató: {clustering_result['silhouette']:.4f}")